# Prática complementar: autoencoder convolucional no MNIST

Na [prática guiada](2_pratica_guiada_mnist.ipynb) a CNN aprendeu a **classificar**: para cada
imagem, um rótulo que alguém anotou à mão. Aqui a tarefa muda de natureza. A rede vai aprender a
**reconstruir a própria imagem**, e o "rótulo" é a imagem original. Nenhuma anotação humana
envolvida. É o que se chama de treino **self-supervised** (auto-supervisionado): a supervisão sai
do próprio dado.

### A teoria em poucas linhas

Um **autoencoder** é uma rede em forma de ampulheta, feita de duas metades:

- **Encoder**: comprime a imagem até uma representação pequena, o **espaço latente**
  (ou *bottleneck*, o gargalo).
- **Decoder**: parte dessa representação comprimida e tenta reconstruir a imagem de entrada.

A loss é a diferença entre entrada e reconstrução, pixel a pixel. Como o gargalo é *menor* que a
imagem, a rede não pode simplesmente copiar os pixels: para reconstruir bem, ela precisa guardar no
latente o que **importa** (a forma do traço) e descartar o resto. É compressão aprendida, e a
representação que sobra no meio é útil bem além da reconstrução (seção 6).

**Denoising autoencoder** é a variante que vamos treinar: a entrada é corrompida com ruído e o alvo
continua sendo a imagem limpa. Sujar a entrada impede que a rede aprenda simplesmente a função
identidade: ela é obrigada a modelar a estrutura do dígito. E de graça ganhamos um modelo que
limpa imagens sujas.

Pipeline:

1. **Dados**: o MNIST, agora sem rótulos.
2. **Ruído como camada**: uma camada de augmentation corrompe a entrada *on the fly*.
3. **Arquitetura**: encoder convolucional + decoder com `Conv2DTranspose`.
4. **Treino**: o alvo é a própria entrada.
5. **Inferência**: o modelo **salvo** limpando os dígitos ruidosos do conjunto de teste.
6. **Encoder como feature extractor** *(complementar)*.

## Setup

In [ ]:
"""
Define uma seed global para garantir reprodutibilidade máxima
no treinamento com TensorFlow/Keras.

NOTE 1: Diferente da prática guiada (onde deixo essa parte comentada para
        demonstrar o comportamento estocástico do DL), aqui as seeds ficam
        ativas: você roda sozinha(o), e é melhor que os números batam
        com o que está escrito nos comentários.
NOTE 2: Na prática diária com DL, determinamos as SEEDs somente no
        split dos dados, pois operações em GPU sofrem perda de
        performance quando forçadas ao determinismo.
"""
# 1. Variáveis de ambiente (precisam ser definidas ANTES de importar TF!)
import os
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'      # força kernels determinísticos
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'    # determinismo em GPU (cuDNN)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from absl import logging

# seeds determinadas após importação dos módulos
keras.utils.set_random_seed(SEED)
# 3. Habilita operações determinísticas no TF (TF >= 2.8)
# NOTE: Haverá perda de performance
tf.config.experimental.enable_op_determinism()

# silenciar alguns avisos do tf que não serão relevantes
tf.get_logger().setLevel('ERROR')
logging.set_verbosity(logging.ERROR)

### Funções auxiliares

In [ ]:
def plot_images(images, titles=None, scale=3, vrange=(0, 1)):
    """Exibe uma lista de imagens lado a lado.

    vrange: faixa fixa de intensidade. O padrão (0, 1) é a escala do MNIST
            normalizado, e é o que deixa original, versão ruidosa e reconstrução
            comparáveis entre si. Use vrange=None para cada imagem se
            auto-escalar (útil nos feature maps da seção 6).
    """
    n = len(images)
    vmin, vmax = vrange if vrange is not None else (None, None)
    fig, axes = plt.subplots(1, n, figsize=(scale * n, scale))
    axes = np.atleast_1d(axes)  # garante iterável quando há uma única imagem

    for i, (ax, img) in enumerate(zip(axes, images)):
        # squeeze remove o eixo de canal: (28, 28, 1) -> (28, 28)
        ax.imshow(np.squeeze(np.asarray(img)), cmap="gray", vmin=vmin, vmax=vmax)
        ax.axis("off")
        if titles is not None:
            ax.set_title(titles[i], fontsize=12)

    plt.tight_layout()
    plt.show()


def plot_history(history, skip_epochs=0, smooth=1):
    """Plota as curvas de treino e validação para cada métrica registrada.

    history: o objeto devolvido por model.fit (ou um dict/DataFrame equivalente)
    skip_epochs: ignora as primeiras épocas, cuja escala costuma achatar o resto do gráfico
    smooth: janela da média móvel; 1 desliga a suavização
    """
    df = history.history if hasattr(history, "history") else history
    df = pd.DataFrame(df).iloc[skip_epochs:]

    if smooth > 1:
        janela = np.ones(smooth) / smooth
        df = pd.DataFrame({c: np.convolve(df[c], janela, mode="valid") for c in df.columns})

    # As métricas "base" são as colunas sem o prefixo val_; cada uma vira um subplot
    metricas = [c for c in df.columns if not c.startswith("val_")]

    ncols = 2
    nrows = int(np.ceil(len(metricas) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 4.5), squeeze=False)
    axes = axes.flatten()

    for ax, metrica in zip(axes, metricas):
        ax.plot(df[metrica], linestyle="-", label="treino")
        if f"val_{metrica}" in df.columns:  # nem toda métrica tem par de validação
            ax.plot(df[f"val_{metrica}"], linestyle="--", label="validação")
        ax.set_title(metrica.replace("_", " ").title())
        ax.set_xlabel("Época")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.7)

    # Descarta os subplots que sobraram na última linha
    for ax in axes[len(metricas):]:
        fig.delaxes(ax)

    plt.tight_layout()
    plt.show()

## 1. Dados

O mesmo MNIST da prática guiada: 70 mil imagens 28x28 em escala de cinza, divididas em treino
(60 mil) e teste (10 mil). A diferença é que os **rótulos não entram no treino**: eles são
carregados apenas para escrever os títulos dos gráficos.

In [ ]:
# y_train / y_test aparecem só nos títulos das figuras: o treino não usa rótulo nenhum!
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print(f"Treino: {x_train.shape} imagens")
print(f" Teste: {x_test.shape} imagens")
print(f"Faixa de valores dos pixels: {x_train.min()} a {x_train.max()} ({x_train.dtype})")

### Pré-processamento

Duas coisas, como antes:

- **Eixo de canal**: `(N, 28, 28)` -> `(N, 28, 28, 1)`, o formato que as camadas do Keras esperam.
- **Escala**: pixels de 0-255 para 0-1.

Uma mudança em relação à prática guiada: lá o `Rescaling` era uma **camada dentro do modelo**.
Aqui não. O motivo é a própria natureza do autoencoder: o **alvo é a imagem**, e o alvo não passa
pelo modelo, ele vai direto para a loss. Como a saída do decoder é uma `sigmoid` (que só produz
valores entre 0 e 1), o alvo precisa estar nessa mesma faixa. Normalizar os dados uma vez, fora do
modelo, deixa entrada e alvo na mesma escala sem truque nenhum.

In [ ]:
# (60000, 28, 28) -> (60000, 28, 28, 1), já em float 0-1
x_train = (x_train.reshape(-1, 28, 28, 1) / 255.0).astype("float32")
x_test = (x_test.reshape(-1, 28, 28, 1) / 255.0).astype("float32")

INPUT_SHAPE = x_train.shape[1:]  # (28, 28, 1)

print(f"x_train: {x_train.shape}  ({x_train.min():.1f} a {x_train.max():.1f})")
print(f" x_test: {x_test.shape}")
print(f"INPUT_SHAPE: {INPUT_SHAPE}")

In [ ]:
# Amostra aleatória do conjunto de treino
idx = np.random.choice(len(x_train), 8, replace=False)
plot_images(x_train[idx], titles=[f"Rótulo: {y}" for y in y_train[idx]])

## 2. O ruído como camada de augmentation

A camada `GaussianNoise(stddev)` soma a cada pixel um valor sorteado de uma normal de média 0 e
desvio `stddev`. No catálogo do Keras ela aparece entre as camadas de *regularização*, mas o papel
dela aqui é de **augmentation**: é ela que fabrica a entrada corrompida da nossa tarefa.

Ela é uma camada como qualquer outra e vive dentro da arquitetura, o que nos dá as mesmas três
vantagens da prática guiada:

- **On the fly**: o ruído é sorteado na GPU, batch a batch. A cada época a mesma imagem aparece
  com um ruído diferente, e o par *(sujo, limpo)* sai de graça, sem gerar nem armazenar um
  dataset ruidoso em disco.
- **Só no treino**: a camada age apenas quando `training=True`. Em `evaluate()` e `predict()` ela
  é transparente e deixa a imagem passar intacta. Por isso, na seção 5, vamos **chamar a camada à
  mão** para fabricar o conjunto de teste ruidoso.
- **Acompanha o modelo**: ao salvar e recarregar, o ruído vai junto dentro do arquivo `.keras`,
  e não fica espalhado por um pipeline externo (segurança para reprodutibilidade).

### Só ruído, nada de augmentation "padrão"

Nenhum `RandomTranslation`, `RandomRotation` ou `RandomZoom` aqui, de propósito: a tarefa já é nova
o suficiente, e cada transformação extra dificulta a reconstrução e turva a leitura do resultado.
O único augmentation é o ruído, que neste caso **é a própria definição da tarefa**, não um extra.

> **Para estudar depois:** as camadas de augmentation do Keras cobrem o básico. Quando a tarefa
> exigir mais (ruídos variados, blur, distorções elásticas, *cutout*, recortes com bounding boxes
> ou máscaras de segmentação acompanhando a imagem), o módulo a estudar é o
> **[AlbumentationsX](https://github.com/albumentations-team/AlbumentationsX)**, sucessor mantido
> do Albumentations. São pipelines declarativos, muito rápidos em CPU, independentes de framework
> (Keras, PyTorch, etc.) e o padrão de mercado em visão computacional.

In [ ]:
NOISE_STDDEV = 0.3   # desvio padrão do ruído, na escala 0-1 dos pixels

noise_augmentation = keras.Sequential(
    [
        keras.layers.GaussianNoise(stddev=NOISE_STDDEV, seed=SEED),
    ],
    name="noise_augmentation_module",
)

### Vendo o efeito antes de treinar

Como sempre: antes de colocar a camada no modelo, vale **olhar** o que ela produz. É a forma mais
rápida de perceber que o ruído está fraco demais (tarefa trivial) ou forte demais (dígito
irreconhecível, e aí nem nós nem a rede conseguimos reconstruir).

Repare no `training=True`: é ele que liga o ruído. Sem isso, a camada devolveria a imagem original.

In [ ]:
image = x_train[np.random.choice(x_train.shape[0])]

# A camada espera um batch; repetimos a mesma imagem 6 vezes para ver 6 sorteios diferentes
batch = np.repeat(image[None, ...], 6, axis=0)
noisy = noise_augmentation(batch, training=True).numpy()

print(f"faixa de valores após o ruído: {noisy.min():.2f} a {noisy.max():.2f}")

# O ruído empurra alguns pixels para fora de 0-1. Cortamos APENAS para visualizar;
# o modelo recebe os valores como estão (nem no treino, nem no teste há clip).
plot_images([image], titles=["Original"])
plot_images(np.clip(noisy, 0, 1), titles=[f"Ruído {i + 1}" for i in range(len(noisy))])

## 3. Arquitetura: encoder + decoder

O **encoder** é o bloco de sempre, `Conv2D -> ReLU -> MaxPooling`, reduzindo a resolução
28x28 -> 14x14 -> 7x7 enquanto ganha filtros (32 -> 64). No fim dele, uma `Conv2D` com poucos
filtros faz o aperto final: o **latente** de 7x7x8 = **392 valores**, metade dos 784 pixels da
imagem original. É esse estrangulamento que impede a rede de copiar a entrada.

O **decoder** precisa do caminho inverso: *aumentar* a resolução de volta para 28x28.

### `Conv2DTranspose`, a camada que ainda não vimos

`Conv2DTranspose(filtros, (3, 3), strides=2)` é a operação espelhada da `Conv2D`:

- A `Conv2D` **resume** uma vizinhança num pixel: a janela desliza sobre a entrada e cada posição
  produz um único número.
- A `Conv2DTranspose` **espalha** um pixel numa vizinhança: cada valor da entrada multiplica o
  kernel inteiro e o resultado é *depositado* (somado) numa janela da saída. Com `strides=2` os
  depósitos ficam de 2 em 2 pixels **na saída**, que por isso sai com o dobro do tamanho.

Ou seja: é um "carimbo" cujos pesos são aprendidos por *gradient descent*, igual à `Conv2D`.
Você vai encontrá-la chamada de *deconvolução* (nome tecnicamente errado, ela não é a inversa
matemática da convolução) ou de *upconvolução*.

A última camada é uma `Conv2D(1, activation="sigmoid")`: **um** canal de saída (a imagem em escala
de cinza) com valores esmagados para 0-1, a mesma faixa do alvo.

In [ ]:
keras.utils.set_random_seed(SEED)

inputs = keras.Input(shape=INPUT_SHAPE)

# Augmentation: corrompe a entrada. Ativo APENAS durante o treino.
x = noise_augmentation(inputs)

# --- Encoder: 28x28x1 -> 7x7x8 ------------------------------------------------
x = keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv1")(x)
x = keras.layers.MaxPooling2D()(x)   # 28x28 -> 14x14

x = keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2")(x)
x = keras.layers.MaxPooling2D()(x)   # 14x14 -> 7x7

# O gargalo: 7x7x8 = 392 valores (a imagem tem 784 pixels)
latent = keras.layers.Conv2D(8, (3, 3), activation="relu", padding="same", name="latent")(x)

# --- Decoder: 7x7x8 -> 28x28x1 ------------------------------------------------
x = keras.layers.Conv2DTranspose(64, (3, 3), strides=2, activation="relu",
                                 padding="same", name="deconv1")(latent)   # 7x7 -> 14x14
x = keras.layers.Conv2DTranspose(32, (3, 3), strides=2, activation="relu",
                                 padding="same", name="deconv2")(x)        # 14x14 -> 28x28

# 1 canal, sigmoid: a imagem reconstruída, na mesma faixa 0-1 do alvo
outputs = keras.layers.Conv2D(1, (3, 3), activation="sigmoid",
                              padding="same", name="reconstruction")(x)

autoencoder = keras.Model(inputs, outputs, name="autoencoder_mnist")

autoencoder.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    # Não há classes aqui: a loss compara a reconstrução com o alvo, pixel a pixel.
    # BCE porque os pixels estão em 0-1 e podem ser lidos como "probabilidade
    # de tinta": é o par natural da sigmoid da última camada (ver nota abaixo)
    loss=keras.losses.BinaryCrossentropy(),
    # MAE mede o mesmo erro em unidades de pixel: bem mais fácil de interpretar
    metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
)

autoencoder.summary()

### A loss: por que `BinaryCrossentropy`, e não MSE

O reflexo natural para "comparar duas imagens" é o erro quadrático médio (MSE). Aqui ele
**não funciona**: combinado com a `sigmoid` da saída, os gradientes saturam e este autoencoder
simplesmente não sai do lugar. A loss trava na primeira época e a rede passa a devolver a mesma
imagem cinza para qualquer entrada. É um treino **colapsado**, e vale provocar esse erro de
propósito para aprender a reconhecer o sintoma: loss constante e reconstrução sempre idêntica.

Como os pixels estão em 0-1, podemos lê-los como "probabilidade de ter tinta" e usar a
`BinaryCrossentropy` pixel a pixel: é o par natural da sigmoid e o padrão para autoencoders de
MNIST. Em troca, o valor da loss perde interpretação visual, por isso a `MeanAbsoluteError`
entra como métrica: MAE de 0.02 significa "erro médio de 2% de intensidade por pixel".

**Experimente** (depois de rodar o notebook inteiro): troque a loss por
`keras.losses.MeanSquaredError()`, retreine e veja o colapso acontecer.

Leia o `summary()` de cima para baixo acompanhando a coluna *Output Shape*: a ampulheta está toda
ali, 28x28 -> 14x14 -> 7x7 -> 14x14 -> 28x28. Repare também no total de parâmetros, bem menos que
a CNN classificadora da prática guiada, porque não há nenhuma camada `Dense`.

## 4. Treino

A linha que resume o self-supervised é esta:

> `autoencoder.fit(x_train, x_train, ...)`

A mesma variável entra como **entrada** e como **alvo**. Não existe `y`. O ruído aparece depois, já
dentro do grafo: a camada suja a entrada, e o alvo continua limpo.

### Uma sutileza importante na validação

A camada de ruído só age em `training=True`. Logo:

- **treino**: entrada suja -> alvo limpo (*denoising* de verdade).
- **validação**: entrada limpa -> alvo limpo (reconstrução pura, um problema mais fácil).

Por isso não espere as curvas da prática guiada. Aqui a `val_loss` costuma ficar **abaixo** da loss
de treino, e isso não é bug: são tarefas diferentes sendo medidas. O `EarlyStopping` acaba
monitorando a qualidade de reconstrução limpa, que funciona bem como proxy. Se quiser medir
denoising na validação, o caminho é fabricar um conjunto de validação ruidoso **fixo** (como
faremos com o teste na seção 5) e passá-lo em `validation_data=(x_val_noisy, x_val)`.

In [ ]:
BATCH_SIZE = 256
EPOCHS = 50          # teto; Talvez haja margem para treinos mais longos...
VALIDATION_SPLIT = 0.2
MODEL_PATH = "models/autoencoder_mnist.keras"

os.makedirs("models", exist_ok=True)   # o ModelCheckpoint não cria a pasta sozinho


def make_callbacks(patience=5, checkpoint=MODEL_PATH):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint,
            monitor="val_loss",
            save_best_only=True,
        )
    ]


def train(model, patience=5, checkpoint=MODEL_PATH, verbose=1):
    """Treina um autoencoder com a configuração padrão e devolve o history."""
    history = model.fit(
        x_train, x_train,          # entrada E alvo: a supervisão vem do próprio dado
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_split=VALIDATION_SPLIT,
        callbacks=make_callbacks(patience, checkpoint),
        verbose=verbose,
    )
    print("\nTreinamento concluído!")
    return history

O treino inteiro leva cerca de 1 minuto em GPU; em CPU, alguns minutos. Se estiver
lento demais na sua máquina, baixe o teto de `EPOCHS` ou treine com um subconjunto
(`x_train[:10000]`): o resultado fica pior, mas o notebook roda inteiro.

In [ ]:
history = train(autoencoder)

In [ ]:
plot_history(history, skip_epochs=1, smooth=0)

## 5. Denoising com o modelo salvo

O `ModelCheckpoint` guardou a melhor época em `models/autoencoder_mnist.keras`. Vamos recarregar
**o arquivo**, e não usar o objeto que já está na memória: é assim na vida real, o treino acontece
num lugar e a inferência em outro.

Repare no `summary()` do modelo carregado: a camada de ruído **veio junto**. O augmentation é parte
do modelo, não de um script externo que alguém precisa lembrar de reproduzir.

In [ ]:
autoencoder_salvo = keras.models.load_model(MODEL_PATH)
autoencoder_salvo.summary()

### Fabricando o conjunto de teste ruidoso

Para demonstrar o denoising precisamos do par *(sujo, limpo)* no teste. Mas a camada de ruído é
transparente em inferência, ela não vai sujar nada sozinha durante o `predict()`. Então chamamos a
camada **à mão** com `training=True`, exatamente como na seção 2, só que agora pegando-a de dentro
do modelo carregado (`get_layer`). Assim garantimos que o ruído do teste é o mesmo ruído do treino.

In [ ]:
noise_layer = autoencoder_salvo.get_layer("noise_augmentation_module")

x_test_noisy = noise_layer(x_test, training=True).numpy()

print(f"x_test:       {x_test.shape}  ({x_test.min():.2f} a {x_test.max():.2f})")
print(f"x_test_noisy: {x_test_noisy.shape}  ({x_test_noisy.min():.2f} a {x_test_noisy.max():.2f})")

### A inferência

Agora é um `predict()` comum: entra a imagem suja, sai a reconstrução. A camada de ruído está
desligada (estamos em modo de inferência), então o modelo recebe exatamente o `x_test_noisy` que
fabricamos, sem ruído adicional.

In [ ]:
x_test_denoised = autoencoder_salvo.predict(x_test_noisy, verbose=0)

print(f"entrada ruidosa: {x_test_noisy.shape}")
print(f"reconstrução:    {x_test_denoised.shape}")

In [ ]:
n = 8
titles = [f"dígito {y}" for y in y_test[:n]]

plot_images(x_test[:n], titles=titles, scale=2)                       # o que queremos recuperar
plot_images(np.clip(x_test_noisy[:n], 0, 1), titles=["ruidoso"] * n, scale=2)   # o que a rede vê
plot_images(x_test_denoised[:n], titles=["reconstruído"] * n, scale=2)         # o que ela devolve

### Medindo o resultado

A pergunta é: a reconstrução ficou mais perto da imagem limpa do que a própria imagem suja?
O *baseline* a bater é o erro de **não fazer nada**, ou seja, devolver a entrada ruidosa
como resposta.

In [ ]:
# evaluate roda em modo de inferência (ruído desligado): entrada suja, alvo limpo
loss, mae = autoencoder_salvo.evaluate(x_test_noisy, x_test, verbose=0)

# Baseline: erro de entregar a imagem suja como "resposta"
mae_ruidoso = np.abs(x_test_noisy - x_test).mean()

resultados = pd.DataFrame(
    {
        "saída": ["imagem ruidosa (baseline)", "reconstrução do autoencoder"],
        "MAE vs. imagem limpa": [mae_ruidoso, mae],
    }
)
print(f"loss (BCE) no teste: {loss:.4f}\n")
resultados

Duas observações sobre o resultado:

- O MAE da reconstrução deve ficar **bem abaixo** do baseline: o modelo aprendeu a jogar o ruído
  fora. E note que ele nunca viu um único rótulo.
- As reconstruções saem visivelmente **mais lisas** que os originais. O decoder parte de 392
  números e não tem como recuperar as microvariações do traço; esse borrão é a assinatura da
  informação perdida no gargalo. É também a razão pela qual autoencoders simples como este não
  servem para *gerar* imagens nítidas (esse é o território dos VAEs, GANs e modelos de difusão).

**Experimente**: suba o `NOISE_STDDEV` para 0.5 ou 0.8 e retreine. Em algum ponto o dígito se
perde no ruído e a rede passa a devolver um "dígito médio" borrado: informação que não existe na
entrada não pode ser recuperada. Depois tente o contrário: aumente o gargalo (`latent` com 32
filtros em vez de 8) e veja as reconstruções ficarem mais nítidas.

## 6. O encoder como feature extractor - Seção complementar

Treinamos o autoencoder para reconstruir imagens, mas na maioria das aplicações reais **a
reconstrução é descartada**. O produto que interessa é o *encoder*: uma função que transforma
784 pixels em 392 números que descrevem a *estrutura* do dígito.

O recorte é o mesmo truque da seção 5 da prática guiada: um modelo funcional do Keras é um grafo,
então podemos declarar um modelo novo que reaproveita a entrada e termina numa camada intermediária.
Nenhum peso é copiado ou retreinado.

In [ ]:
def make_feature_extractor(model, layer_name):
    """Recorta um modelo treinado, transformando uma camada interna na saída."""
    return keras.Model(
        inputs=model.input,                          # mesma entrada do modelo original
        outputs=model.get_layer(layer_name).output,  # saída de uma camada específica
        name=f"fe_{layer_name}",
    )


encoder = make_feature_extractor(autoencoder_salvo, "latent")
encoder.summary()

In [ ]:
sample = x_test[:8]
z = encoder.predict(sample, verbose=0)

print(f"entrada: {sample.shape} -> {np.prod(sample.shape[1:])} pixels por imagem")
print(f"latente: {z.shape} -> {np.prod(z.shape[1:])} valores por imagem")

Os 8 mapas do latente, para uma imagem: cada um é uma grade 7x7 que responde a um padrão diferente.
Aqui usamos `vrange=None` para que cada mapa se auto-escale, porque as ativações da ReLU não
estão limitadas a 0-1 como os pixels.

Note que passamos a imagem **limpa** e o encoder a aceita sem reclamar: a camada de ruído continua
no grafo, mas em inferência ela é transparente.

In [ ]:
img_idx = 0

# (7, 7, 8) -> (8, 7, 7): os canais do latente passam a ser "imagens"
maps = np.transpose(z[img_idx], [2, 0, 1])

plot_images([sample[img_idx]], titles=[f"Entrada: dígito {y_test[img_idx]}"])
plot_images(maps, titles=[f"latente {i}" for i in range(len(maps))], scale=1.8, vrange=None)

Em 7x7 não há mais nada "reconhecível": o latente não é uma imagem, é uma **assinatura** de padrões,
e é justamente por ser abstrato que ele é útil.

E é esse o raciocínio por trás do **pré-treino self-supervised**, que sustenta boa parte do DL
moderno: treina-se um modelo numa tarefa que não exige rótulo (reconstruir a entrada, prever o
pedaço que falta, prever a próxima palavra...) sobre um monte de dados crus, e depois se
aproveita o encoder na tarefa de interesse, que tem poucos rótulos.

Para ver isso funcionando, o caminho é curto: troque o decoder por
`Flatten -> Dense(10, activation="softmax")`, congele o encoder (`encoder.trainable = False`) e
treine essa cabeça de classificação com **1000** imagens rotuladas. Compare com a CNN da prática
guiada treinada do zero com as mesmas 1000 imagens. É aí que a vantagem do pré-treino aparece.

---

### Exercícios propostos

- Aperte o gargalo: `latent` com 4 ou 2 filtros (196 ou 98 valores, contra 784 pixels). Até onde
  dá para comprimir antes de o dígito virar um borrão genérico?
- Faça a validação medir *denoising* de verdade: separe um `x_val` à mão, gere o `x_val_noisy` uma
  única vez (como na seção 5) e troque o `validation_split` por `validation_data=(x_val_noisy, x_val)`.
  Compare o formato das curvas com o que você viu aqui.
- Troque o tipo de corrupção: no lugar do `GaussianNoise`, use `keras.layers.RandomErasing`, que
  apaga um retângulo da imagem (e se comporta igual, transparente em inferência). A rede é a mesma,
  mas a tarefa vira *inpainting*: reconstruir o pedaço que falta.
- Passe imagens do Fashion MNIST (`keras.datasets.fashion_mnist`) pelo denoiser treinado em dígitos.
  Ele limpa a roupa, ou tenta transformá-la em dígito? A resposta diz muito sobre o que o encoder
  aprendeu.
- O exercício grande é o do fim da seção 6: congelar o encoder, plugar uma cabeça de classificação
  e treinar com 1000 rótulos.
